importando as bibliotecas, os dados e tratando

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import mlflow

df = pd.read_csv("../data/telco_churn_clean.csv")

drop_cols = ['CustomerID', 'Country', 'State', 'City', 'Lat Long', 
             'Churn Reason', 'Churn Score', 'Churn Value', 'CLTV']

X = df.drop(columns=drop_cols + ['Churn Label'])
y = (df['Churn Label'] == 'Yes').astype(int)
X = pd.get_dummies(X)


Dividindo entre treino e teste

In [2]:
# Dividir em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

Normalizando

In [3]:
# Normalizar
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Convertendo para tensores PyTorch (PyTorch trabalha apenas com seu próprio formato, o tensor, matriz otimizada para rodar em GPU)

In [ ]:
X_train_t = torch.FloatTensor(X_train)
X_test_t = torch.FloatTensor(X_test)
y_train_t = torch.FloatTensor(y_train.values)
y_test_t = torch.FloatTensor(y_test.values)
#Divisao Treino 80% Teste 20%
print(f"Shape treino: {X_train_t.shape}")
print(f"Shape teste: {X_test_t.shape}")

Shape treino: torch.Size([5634, 50])
Shape teste: torch.Size([1409, 50])


Construindo Arquitetura da Rede Neural

In [ ]:
class ChurnMLP(nn.Module): #Criando rede neural como classe
    def __init__(self, input_dim #Numero de features
    ): 
        super().__init__()
        self.network = nn.Sequential( #Define as camadas em que os dados passarão em sequencia
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1), #retorna 1 unico valor
            nn.Sigmoid() #aqui vira probabilidade
        ) 
    
    def forward(self, x):
        return self.network(x)

model = ChurnMLP(input_dim=X_train_t.shape[1])
print(model) #print da estrutura

ChurnMLP(
  (network): Sequential(
    (0): Linear(in_features=50, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
    (7): Sigmoid()
  )
)


Otimizador e função de perda

In [6]:
# Por termos um desbalanceamento nos dados, com 5174 ''não cancela'' e 1869''cancela'' estou adicionando weight para a rede aprender que errar um churn é mais grave do que errar um não churn
pos_weight = torch.tensor([len(y_train[y_train==0]) / len(y_train[y_train==1])])

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Peso da classe positiva: {pos_weight.item():.2f}")

Peso da classe positiva: 2.77


Loop de treinamento com Early Stopping ( Condição para que se o modelo não melhorar, ele pare de treinar)

In [7]:
epochs = 100
patience = 10  # para se não melhorar em 10 épocas seguidas
best_loss = float('inf')
patience_counter = 0

train_losses = []

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    outputs = model(X_train_t).squeeze()
    loss = criterion(outputs, y_train_t)
    
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item())
    
    # Early stopping
    if loss.item() < best_loss:
        best_loss = loss.item()
        patience_counter = 0
        torch.save(model.state_dict(), "../models/mlp_best.pt")
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f"Early stopping na época {epoch+1}")
        break
    
    if (epoch+1) % 10 == 0:
        print(f"Época {epoch+1} | Loss: {loss.item():.4f}")

Época 10 | Loss: 1.0337
Época 20 | Loss: 1.0041
Época 30 | Loss: 0.9723
Época 40 | Loss: 0.9469
Época 50 | Loss: 0.9263
Época 60 | Loss: 0.9163
Época 70 | Loss: 0.9087
Época 80 | Loss: 0.9051
Época 90 | Loss: 0.9011
Época 100 | Loss: 0.9002


Como rodou todas as 100 épocas, significa que o modelo continuou melhorando e não precisou do early stopping, vou adicionar mais épocas para explorarmos melhor.

In [8]:
epochs = 300
patience = 10  # para se não melhorar em 10 épocas seguidas
best_loss = float('inf')
patience_counter = 0

train_losses = []

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    outputs = model(X_train_t).squeeze()
    loss = criterion(outputs, y_train_t)
    
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item())
    
    # Early stopping
    if loss.item() < best_loss:
        best_loss = loss.item()
        patience_counter = 0
        torch.save(model.state_dict(), "../models/mlp_best.pt")
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f"Early stopping na época {epoch+1}")
        break
    
    if (epoch+1) % 10 == 0:
        print(f"Época {epoch+1} | Loss: {loss.item():.4f}")

Época 10 | Loss: 0.8981
Época 20 | Loss: 0.8985
Early stopping na época 25


Avaliando o modelo. 
Fazer previsões no conjunto de teste e calcular AUC e F1 para comparar com baseline

In [12]:
from sklearn.metrics import roc_auc_score, f1_score, classification_report

with mlflow.start_run(run_name="MLP-v1"):
    mlflow.log_param("hidden_layers", [64, 32])
    mlflow.log_param("dropout", 0.3)
    mlflow.log_param("lr", 0.001)
    mlflow.log_param("patience", 10)
    
    mlflow.log_metric("auc_roc", 0.830)
    mlflow.log_metric("f1", 0.621)
model.eval()
with torch.no_grad():
    preds = model(X_test_t).squeeze()
    preds_binary = (preds >= 0.5).float()

auc = roc_auc_score(y_test, preds.numpy())
f1 = f1_score(y_test, preds_binary.numpy())

print("=" * 45)
print(f"{'Modelo':<25} {'AUC':>8} {'F1':>8}")
print("=" * 45)
print(f"{'Dummy':<25} {'0.500':>8} {'0.000':>8}")
print(f"{'Logistic Regression':<25} {'0.856':>8} {'0.619':>8}")
print(f"{'MLP PyTorch':<25} {auc:>8.3f} {f1:>8.3f}")
print("=" * 45)
print("\n", classification_report(y_test, preds_binary.numpy(), target_names=['Ficou', 'Cancelou']))

Modelo                         AUC       F1
Dummy                        0.500    0.000
Logistic Regression          0.856    0.619
MLP PyTorch                  0.830    0.621

               precision    recall  f1-score   support

       Ficou       0.88      0.81      0.84      1035
    Cancelou       0.57      0.69      0.62       374

    accuracy                           0.78      1409
   macro avg       0.72      0.75      0.73      1409
weighted avg       0.79      0.78      0.78      1409



### Leitura dos dados

**Precision** - 57% dos que o modelo disse que iriam cancelar realmente cancelaram

**Recall** - 69% dos que cancelaram, o modelo identificou

**F1** - Equilibrio entre os dois - 62%

### Ajustando hiperparâmetros para melhorar o modelo

In [16]:
mlflow.end_run()
mlflow.set_tracking_uri("sqlite:///../mlruns/mlflow.db")
mlflow.set_experiment("telco-churn")

with mlflow.start_run(run_name="MLP-v2"):
    # Hiperparâmetros
    mlflow.log_param("hidden_layers", [128, 64, 32])
    mlflow.log_param("dropout", 0.2)
    mlflow.log_param("lr", 0.001)
    mlflow.log_param("patience", 20)
    
    # Treino
    model = ChurnMLP(input_dim=X_train_t.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    epochs = 300
    patience = 20
    best_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train_t).squeeze()
        loss = criterion(outputs, y_train_t)
        loss.backward()
        optimizer.step()

        if loss.item() < best_loss:
            best_loss = loss.item()
            patience_counter = 0
            torch.save(model.state_dict(), "../models/mlp_best.pt")
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping na época {epoch+1}")
            break

        if (epoch+1) % 10 == 0:
            print(f"Época {epoch+1} | Loss: {loss.item():.4f}")

    # Avaliação
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t).squeeze()
        preds_binary = (preds >= 0.5).float()

    auc = roc_auc_score(y_test, preds.numpy())
    f1 = f1_score(y_test, preds_binary.numpy())

    mlflow.log_metric("auc_roc", auc)
    mlflow.log_metric("f1", f1)

    print(f"\nMLP v2 — AUC: {auc:.3f} | F1: {f1:.3f}")

Época 10 | Loss: 1.0427
Época 20 | Loss: 1.0076
Época 30 | Loss: 0.9698
Época 40 | Loss: 0.9432
Época 50 | Loss: 0.9250
Época 60 | Loss: 0.9153
Época 70 | Loss: 0.9079
Época 80 | Loss: 0.9053
Época 90 | Loss: 0.9029
Época 100 | Loss: 0.9002
Época 110 | Loss: 0.8972
Época 120 | Loss: 0.8967
Época 130 | Loss: 0.8977
Época 140 | Loss: 0.8964
Época 150 | Loss: 0.8941
Época 160 | Loss: 0.8929
Época 170 | Loss: 0.8931
Época 180 | Loss: 0.8923
Época 190 | Loss: 0.8909
Early stopping na época 197

MLP v2 — AUC: 0.835 | F1: 0.620
